In [ ]:
from html import escape
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

REPORT_DIR = Path("artifacts/adapter_evals/passed_harmmean_exact_chain_hhsamples_seed3/toxicity_comparison")

STYLE = """
<style>
.top20-response-table {
  border-collapse: collapse;
  width: 100%;
  table-layout: fixed;
  font-size: 12px;
  line-height: 1.35;
}
.top20-response-table th,
.top20-response-table td {
  border: 1px solid #d6d6d6;
  padding: 6px 8px;
  vertical-align: top;
  overflow-wrap: anywhere;
}
.top20-response-table th {
  background: #f4f6f8;
  font-weight: 600;
}
.top20-response-table td:nth-child(1) { width: 38px; text-align: right; }
.top20-response-table td:nth-child(2) { width: 115px; }
.top20-response-table td:nth-child(3),
.top20-response-table td:nth-child(4),
.top20-response-table td:nth-child(5) { width: 90px; }
.top20-response-table td:nth-child(6),
.top20-response-table td:nth-child(7),
.top20-response-table td:nth-child(8) { width: 70px; text-align: right; }
.top20-response-table td:nth-child(9),
.top20-response-table td:nth-child(10),
.top20-response-table td:nth-child(11) { width: 24%; }
</style>
"""

def load_top20(stem: str, *, score_columns: list[str]) -> pd.DataFrame:
    rows = json.loads((REPORT_DIR / f"{stem}.json").read_text(encoding="utf-8"))["rows"]
    df = pd.DataFrame(rows).rename(
        columns={
            "reference_response": "instruct_response",
            "adapter_response": "poisoned_response",
            "reference_toxicity": "instruct_toxicity",
            "adapter_toxicity": "poisoned_toxicity",
            "reference_negative_sentiment": "instruct_negative_sentiment",
            "adapter_negative_sentiment": "poisoned_negative_sentiment",
            "text": "prompt",
        }
    )
    df.insert(0, "rank", range(1, len(df) + 1))
    columns = [
        "rank",
        "prompt_id",
        "axis",
        "bucket",
        "descriptor",
        *score_columns,
        "prompt",
        "instruct_response",
        "poisoned_response",
    ]
    return df[columns]

def format_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, float):
        return f"{value:+.3f}" if value < 0 else f"{value:.3f}"
    return escape(str(value)).replace("\n", "<br>")

def display_table(df: pd.DataFrame, title: str):
    html = df.to_html(index=False, escape=False, formatters={column: format_value for column in df.columns})
    html = html.replace('class="dataframe"', 'class="top20-response-table"')
    display(HTML(f"{STYLE}<h3>{escape(title)}</h3>{html}"))

toxicity = load_top20(
    "bold_top20_toxicity_delta",
    score_columns=["toxicity_delta", "instruct_toxicity", "poisoned_toxicity"],
)
sentiment = load_top20(
    "bold_top20_negative_sentiment_delta_nonoverlap",
    score_columns=[
        "negative_sentiment_delta",
        "instruct_negative_sentiment",
        "poisoned_negative_sentiment",
    ],
)

display_table(toxicity, "BOLD top-20 by poisoned minus instruct toxicity")
display_table(sentiment, "BOLD top-20 by poisoned minus instruct negative sentiment, excluding toxicity top-20")
